# GK-2A 12:00~14:00 단기 시계열 수집 + 14:00 ASOS TA/HM 병합

이 노트북은 다음 작업을 순서대로 수행합니다.

1. Google Drive 연결
2. `SME_DATA` GitHub 브랜치 clone/update
3. 패키지 설치
4. KMA API Key 설정
5. 수집 기간 설정
6. GK-2A 16채널을 **12:00~14:00 KST, 10분 간격(13시각)** 으로 수집
7. 같은 날짜의 **14:00 ASOS TA/HM만 정답 라벨**로 수집
8. 공식 `station_list.csv` 좌표를 이용해 지점별 위성 중심 픽셀 추출
9. LSTM/GRU용 `long` 데이터와 HGB/CatBoost/LightGBM용 `wide` 데이터 생성
10. 결측/실패 현황 확인

> 중요: 12:00~13:50의 ASOS TA/HM은 입력 feature로 사용하지 않습니다.  
> 온도/습도는 14:00 정답 라벨로만 붙입니다.


## 1. Google Drive 연결
결과 파일과 원본 위성 파일을 Drive에 직접 저장합니다.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 2. 실행 설정

처음에는 전체 기간보다 **짧은 기간으로 정상 작동 여부를 확인**하는 것을 권장합니다.
아래 기본값은 2024-08-24 ~ 2024-08-30입니다.


In [ ]:
from pathlib import Path

# ===== 사용자 설정 =====
START_DATE = "2024-08-24"
END_DATE   = "2024-08-30"

START_TIME = "12:00"
END_TIME   = "14:00"
STEP_MINUTES = 10

BRANCH = "agent/shortterm-12to14-pipeline"

# Drive 저장 위치
DRIVE_ROOT = Path("/content/drive/MyDrive/SME_DATA")
OUTPUT_ROOT = DRIVE_ROOT / "processed_station_features" / "shortterm_12to14_data"

# Colab 내 GitHub 저장소 위치
REPO_DIR = Path("/content/SME_DATA")

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print("수집 기간:", START_DATE, "~", END_DATE)
print("수집 시각:", START_TIME, "~", END_TIME, f"/ {STEP_MINUTES}분 간격")
print("Drive 저장:", OUTPUT_ROOT)


## 3. GitHub 코드 가져오기

이미 clone되어 있으면 해당 브랜치 최신 코드로 갱신합니다.


In [ ]:
import os, subprocess, shutil

REPO_URL = "https://github.com/tswaincae1221/SME_DATA.git"

if not REPO_DIR.exists():
    subprocess.run(
        ["git", "clone", "-b", BRANCH, REPO_URL, str(REPO_DIR)],
        check=True
    )
else:
    os.chdir(REPO_DIR)
    subprocess.run(["git", "fetch", "origin"], check=True)
    subprocess.run(["git", "checkout", BRANCH], check=True)
    subprocess.run(["git", "pull", "origin", BRANCH], check=True)

os.chdir(REPO_DIR)
print("현재 위치:", os.getcwd())


## 4. 패키지 설치


In [ ]:
!pip -q install -e .
!pip -q install requests pandas numpy xarray h5netcdf netCDF4 pyyaml


## 5. KMA API Key 설정

가능하면 **Colab Secrets**에 `KMA_API_KEY`라는 이름으로 저장하세요.

- 왼쪽 사이드바의 열쇠(Secrets) 아이콘
- `KMA_API_KEY` 추가
- 아래 셀 실행

Secrets를 사용할 수 없으면 셀 실행 후 직접 입력할 수 있습니다.
입력한 키는 화면에 표시되지 않습니다.


In [ ]:
import os
from getpass import getpass

try:
    from google.colab import userdata
    key = userdata.get("KMA_API_KEY")
except Exception:
    key = None

if not key:
    key = getpass("KMA_API_KEY 입력: " ).strip()

if not key:
    raise ValueError("KMA_API_KEY가 비어 있습니다.")

os.environ["KMA_API_KEY"] = key
print("KMA_API_KEY 설정 완료")


## 6. 필수 파일 확인

`station_list.csv`는 운영진 제공 파일이며 저장소의
`data/metadata/station_list.csv` 위치에 있어야 합니다.


In [ ]:
from pathlib import Path

station_path = REPO_DIR / "data" / "metadata" / "station_list.csv"
print("station_list:", station_path)
print("존재:", station_path.exists())

if not station_path.exists():
    raise FileNotFoundError(
        "data/metadata/station_list.csv가 없습니다. "
        "운영진 제공 station_list.csv를 해당 위치에 넣어주세요."
    )


## 7. 12:00~14:00 위성 + 14:00 ASOS 수집

하루 기준 위성 API 호출 수는:

- 13개 시각 × 16채널 = **208건/일**

이미 정상적으로 저장된 위성 파일과 ASOS 파일은 기본적으로 다시 받지 않으므로,
Colab이 끊겨도 같은 셀을 다시 실행해 이어받을 수 있습니다.


In [ ]:
import subprocess, sys

cmd = [
    sys.executable,
    "scripts/collect_shortterm_12to14.py",
    "--start", START_DATE,
    "--end", END_DATE,
    "--start-time", START_TIME,
    "--end-time", END_TIME,
    "--step-minutes", str(STEP_MINUTES),
    "--output-root", str(OUTPUT_ROOT),
]

print("실행 명령:")
print(" ".join(cmd))
subprocess.run(cmd, cwd=REPO_DIR, check=True)


## 8. 수집 실패 목록 확인

실패가 있어도 정상적으로 받은 파일은 그대로 남습니다.
API 제한/일시 오류가 있었다면 위 수집 셀을 다시 실행하면 기존 파일을 건너뛰며 재시도할 수 있습니다.


In [ ]:
import pandas as pd

failure_path = OUTPUT_ROOT / "outputs" / "shortterm_collection_failures.csv"

if failure_path.exists():
    failures = pd.read_csv(failure_path)
    print("수집 실패 건수:", len(failures))
    display(failures.head(30))
else:
    print("실패 목록 파일이 아직 없습니다.")


## 9. 지점 픽셀 추출 + TA/HM 병합

생성 결과:

- `shortterm_long.csv`
  - 한 행 = `Date × STN_ID × TimeKST`
  - LSTM / GRU용
  - 12:00~14:00 10분 간격의 시계열 유지
  - `TA`, `HM`은 14:00 행에만 존재

- `shortterm_wide.csv`
  - 한 행 = `Date × STN_ID`
  - HGB / CatBoost / LightGBM용
  - 예: `IR112_1200`, `IR112_1210`, ..., `IR112_1400`
  - 마지막에 14:00 `TA`, `HM` 병합

- `shortterm_labels_1400.csv`
  - `Date × STN_ID × TA × HM`

- `shortterm_build_missing.csv`
  - 위성 파일 누락 또는 픽셀 추출 실패 목록


In [ ]:
cmd = [
    sys.executable,
    "scripts/build_shortterm_12to14_dataset.py",
    "--start", START_DATE,
    "--end", END_DATE,
    "--start-time", START_TIME,
    "--end-time", END_TIME,
    "--step-minutes", str(STEP_MINUTES),
    "--input-root", str(OUTPUT_ROOT),
]

print("실행 명령:")
print(" ".join(cmd))
subprocess.run(cmd, cwd=REPO_DIR, check=True)


## 10. 결과 데이터 검수


In [ ]:
dataset_dir = OUTPUT_ROOT / "datasets"

long_path   = dataset_dir / "shortterm_long.csv"
wide_path   = dataset_dir / "shortterm_wide.csv"
labels_path = dataset_dir / "shortterm_labels_1400.csv"
missing_path = dataset_dir / "shortterm_build_missing.csv"

for p in [long_path, wide_path, labels_path, missing_path]:
    print(p.name, "->", p.exists(), f"({p.stat().st_size/1024/1024:.2f} MB)" if p.exists() else "")


In [ ]:
long_df = pd.read_csv(long_path)
wide_df = pd.read_csv(wide_path)
labels_df = pd.read_csv(labels_path)

print("LONG shape :", long_df.shape)
print("WIDE shape :", wide_df.shape)
print("LABEL shape:", labels_df.shape)

print("\nLONG columns:")
print(long_df.columns.tolist())

print("\nWIDE 앞쪽 columns:")
print(wide_df.columns[:30].tolist())

display(long_df.head())
display(wide_df.head())


## 11. 시각 수/지점 수/라벨 병합 검수

10분 간격 12:00~14:00이면 하루·지점당 **13개 timestep**이 정상입니다.


In [ ]:
expected_times = list(range(1200, 1300, 10)) + list(range(1300, 1400, 10)) + [1400]

print("실제 TimeKST:", sorted(long_df["TimeKST"].dropna().astype(int).unique().tolist()))
print("기대 timestep 수:", 13)
print("실제 timestep 종류 수:", long_df["TimeKST"].nunique())
print("지점 수:", long_df["STN_ID"].nunique())
print("날짜 수:", long_df["Date"].nunique())

counts = (
    long_df.groupby(["Date", "STN_ID"])["TimeKST"]
    .nunique()
    .value_counts()
    .sort_index()
)
print("\nDate × STN_ID별 timestep 개수 분포:")
display(counts.rename("count").to_frame())

print("\n14:00 라벨 결측:")
display(
    labels_df[["TA", "HM"]].isna().sum().rename("missing").to_frame()
)


## 12. 위성 채널 결측률 확인


In [ ]:
CHANNELS = [
    "VI004","VI005","VI006","VI008","NR013","NR016","SW038",
    "WV063","WV069","WV073","IR087","IR096","IR105","IR112","IR123","IR133"
]

missing_rate = long_df[CHANNELS].isna().mean().sort_values(ascending=False)
display((missing_rate * 100).rename("missing_%").to_frame())


## 13. HGB용 short-term 파생 feature 예시

`shortterm_wide.csv`를 그대로 사용할 수도 있지만,
추후 아래와 같은 단기 파생값을 만들 수 있습니다.

- 현재값(14:00)
- 1시간 변화량: `14:00 - 13:00`
- 2시간 변화량: `14:00 - 12:00`
- 최근 1~2시간 평균
- 최근 1~2시간 표준편차
- 시간 방향 slope
- min / max / range

이 부분은 실제 모델 실험 단계에서 별도 ablation으로 비교하는 것을 권장합니다.


In [ ]:
# 예시: IR112의 2시간 변화량
example = wide_df[["Date", "STN_ID", "IR112_1200", "IR112_1400", "TA", "HM"]].copy()
example["IR112_diff_2h"] = example["IR112_1400"] - example["IR112_1200"]
display(example.head())


## 14. 전체 기간으로 확장할 때

짧은 기간 검수가 정상적으로 끝난 뒤 `START_DATE`, `END_DATE`만 변경하고
위의 수집/병합 셀을 다시 실행하면 됩니다.

주의:
- 하루 약 208건의 위성 요청이 발생합니다.
- 2019~2025 전체 기간을 한 번에 요청하기보다 연도/월 단위 분할을 권장합니다.
- API 호출 제한이 걸리면 이후 다시 실행해 이어받으면 됩니다.
- 이미 저장된 정상 파일은 자동으로 건너뜁니다.


## 최종 데이터 구조 요약

### LSTM / GRU
```text
Date | STN_ID | TimeKST | LAT | LON | ALT | 16 satellite channels | TA | HM
```

한 날짜·지점에 13개 행이 있으며,
TA/HM은 14:00 시점에만 정답 라벨로 존재합니다.

### HGB / CatBoost / LightGBM
```text
Date | STN_ID | LAT | LON | ALT |
VI004_1200 ... IR112_1400 ... WV073_1400 |
TA | HM
```

한 날짜·지점당 1행으로 펼쳐져 있으므로 tabular 모델에 바로 사용할 수 있습니다.
